In [ ]:
# Import libraries

import os
import json
import random

import numpy as np
import torch
import matplotlib.pyplot as plt

import monai
from monai.data import Dataset, DataLoader, decollate_batch
from monai.networks.nets import UNet
from monai.metrics import DiceMetric
from monai.transforms import (
    LoadImaged,
    EnsureChannelFirstd,
    ConcatItemsd,
    NormalizeIntensityd,
    MapTransform,
    AsDiscrete,
    Compose,
)
from monai.inferers import sliding_window_inference

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"MONAI version: {monai.__version__}")
print(f"Using device: {device}")

In [ ]:
# Config & Paths
# Project paths and settings

BASE_DIR = r"D:\Deep_Projects\brain-tumor-segmentation-3d\repo"

PATHS = {
    "training_dir": os.path.join(BASE_DIR, "data", "brats2020", "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData"),
    "configs": os.path.join(BASE_DIR, "configs"),
    "models": os.path.join(BASE_DIR, "models", "baseline"),
    "figures": os.path.join(BASE_DIR, "results", "figures"),
    "metrics": os.path.join(BASE_DIR, "results", "metrics"),
}

for key in ["figures", "metrics"]:
    os.makedirs(PATHS[key], exist_ok=True)

print("Paths configured:")
for name, path in PATHS.items():
    status = "OK" if os.path.exists(path) else "missing"
    print(f"  [{status}] {name:14s} -> {path}")

In [ ]:
# Load the trained baseline model and prepare the validation dataset

from monai.transforms import MapTransform

class RemapLabeld(MapTransform):
    def __call__(self, data):
        d = dict(data)
        for key in self.keys:
            d[key][d[key] == 4] = 3
        return d


model = UNet(
    spatial_dims=3,
    in_channels=4,
    out_channels=4,
    channels=(16, 32, 64, 128, 256),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

model.load_state_dict(torch.load(os.path.join(PATHS["models"], "best_model.pth")))
model.eval()

print("Model loaded successfully")

# Load the validation split
split_path = os.path.join(PATHS["configs"], "dataset_split.json")
with open(split_path, "r") as f:
    split_dict = json.load(f)

val_patients = split_dict["val"]
modalities = ["t1", "t1ce", "t2", "flair"]
patch_size = (96, 96, 96)

def build_data_dicts(patient_list, training_dir):
    data_dicts = []
    for pid in patient_list:
        patient_dir = os.path.join(training_dir, pid)
        entry = {mod: os.path.join(patient_dir, f"{pid}_{mod}.nii") for mod in modalities}
        entry["label"] = os.path.join(patient_dir, f"{pid}_seg.nii")
        entry["patient_id"] = pid
        data_dicts.append(entry)
    return data_dicts

val_dicts = build_data_dicts(val_patients, PATHS["training_dir"])

val_transforms = Compose([
    LoadImaged(keys=modalities + ["label"]),
    EnsureChannelFirstd(keys=modalities + ["label"]),
    RemapLabeld(keys=["label"]),
    ConcatItemsd(keys=modalities, name="image"),
    NormalizeIntensityd(keys="image", nonzero=True, channel_wise=True),
])

val_ds = Dataset(data=val_dicts, transform=val_transforms)
val_loader = DataLoader(val_ds, batch_size=1, num_workers=0)

print(f"Validation set: {len(val_ds)} patients")

In [ ]:
# Run inference on the full validation set and compute per-patient, per-class Dice

post_pred = AsDiscrete(argmax=True, to_onehot=4)
post_label = AsDiscrete(to_onehot=4)

class_names = {1: "NCR_NET", 2: "ED", 3: "ET"}

results = []

model.eval()
with torch.no_grad():
    for val_data in val_loader:
        pid = val_data["patient_id"][0]
        val_inputs = val_data["image"].to(device)
        val_labels = val_data["label"].to(device)

        val_outputs = sliding_window_inference(val_inputs, patch_size, sw_batch_size=1, predictor=model)

        pred_onehot = post_pred(val_outputs[0])
        label_onehot = post_label(val_labels[0])

        tumor_voxels = int((val_labels > 0).sum().item())

        patient_result = {"patient_id": pid, "tumor_voxels": tumor_voxels}

        # Overall Dice (excluding background)
        overall_dice = DiceMetric(include_background=False, reduction="mean")
        overall_dice(y_pred=pred_onehot.unsqueeze(0), y=label_onehot.unsqueeze(0))
        patient_result["dice_overall"] = overall_dice.aggregate().item()

        # Per-class Dice
        for class_idx, class_name in class_names.items():
            class_dice = DiceMetric(include_background=True, reduction="mean")
            pred_class = pred_onehot[class_idx:class_idx+1].unsqueeze(0)
            label_class = label_onehot[class_idx:class_idx+1].unsqueeze(0)
            class_dice(y_pred=pred_class, y=label_class)
            score = class_dice.aggregate().item()
            patient_result[f"dice_{class_name}"] = score

        results.append(patient_result)
        print(f"{pid}: overall={patient_result['dice_overall']:.3f}, "
              f"NCR/NET={patient_result['dice_NCR_NET']:.3f}, "
              f"ED={patient_result['dice_ED']:.3f}, "
              f"ET={patient_result['dice_ET']:.3f}")

# Save results
metrics_path = os.path.join(PATHS["metrics"], "baseline_per_patient_dice.json")
with open(metrics_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"\nSaved -> {metrics_path}")

In [ ]:
# Analyze the distribution of scores and identify patterns

dice_overall = [r["dice_overall"] for r in results]
dice_ncr = [r["dice_NCR_NET"] for r in results]
dice_ed = [r["dice_ED"] for r in results]
dice_et = [r["dice_ET"] for r in results]

print("Per-class Dice statistics (n=55 validation patients)")
print("-" * 55)
print(f"{'Class':12s} {'mean':>8s} {'median':>8s} {'min':>8s} {'max':>8s}")
print(f"{'Overall':12s} {np.mean(dice_overall):8.3f} {np.median(dice_overall):8.3f} {min(dice_overall):8.3f} {max(dice_overall):8.3f}")
print(f"{'NCR/NET':12s} {np.mean(dice_ncr):8.3f} {np.median(dice_ncr):8.3f} {min(dice_ncr):8.3f} {max(dice_ncr):8.3f}")
print(f"{'ED':12s} {np.mean(dice_ed):8.3f} {np.median(dice_ed):8.3f} {min(dice_ed):8.3f} {max(dice_ed):8.3f}")
print(f"{'ET':12s} {np.mean(dice_et):8.3f} {np.median(dice_et):8.3f} {min(dice_et):8.3f} {max(dice_et):8.3f}")

# Sort by overall Dice to find worst cases
sorted_results = sorted(results, key=lambda x: x["dice_overall"])
worst_5 = sorted_results[:5]
best_5 = sorted_results[-5:]

print(f"\nWorst 5 patients:")
for r in worst_5:
    print(f"  {r['patient_id']}: dice={r['dice_overall']:.3f}, tumor_voxels={r['tumor_voxels']:,}")

print(f"\nBest 5 patients:")
for r in best_5:
    print(f"  {r['patient_id']}: dice={r['dice_overall']:.3f}, tumor_voxels={r['tumor_voxels']:,}")

# Check correlation between tumor size and Dice score
tumor_sizes = [r["tumor_voxels"] for r in results]
correlation = np.corrcoef(tumor_sizes, dice_overall)[0, 1]
print(f"\nCorrelation between tumor size and Dice score: {correlation:.3f}")

In [ ]:
# Visualize the worst-performing patient to understand what went wrong

worst_patient_id = worst_5[0]["patient_id"]
worst_dict = [d for d in val_dicts if d["patient_id"] == worst_patient_id][0]

sample_transformed = val_transforms(worst_dict)
input_volume = sample_transformed["image"].unsqueeze(0).to(device)
label_volume = sample_transformed["label"]

with torch.no_grad():
    pred_volume = sliding_window_inference(input_volume, patch_size, sw_batch_size=1, predictor=model)
    pred_volume = torch.argmax(pred_volume, dim=1).squeeze(0).cpu()

label_np = label_volume.squeeze(0).numpy()
pred_np = pred_volume.numpy()
flair_np = sample_transformed["image"][3].numpy()

tumor_per_slice = (label_np > 0).sum(axis=(0, 1))
best_slice = np.argmax(tumor_per_slice)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(flair_np[:, :, best_slice].T, cmap="gray", origin="lower")
axes[0].set_title("FLAIR")
axes[0].axis("off")

# Ground truth with class-specific colors
gt_slice = label_np[:, :, best_slice].T
axes[1].imshow(flair_np[:, :, best_slice].T, cmap="gray", origin="lower")
gt_masked = np.ma.masked_where(gt_slice == 0, gt_slice)
axes[1].imshow(gt_masked, cmap="jet", alpha=0.6, origin="lower", vmin=0, vmax=3)
axes[1].set_title(f"Ground Truth\n(1=NCR/NET blue, 2=ED green, 3=ET red)")
axes[1].axis("off")

pred_slice = pred_np[:, :, best_slice].T
axes[2].imshow(flair_np[:, :, best_slice].T, cmap="gray", origin="lower")
pred_masked = np.ma.masked_where(pred_slice == 0, pred_slice)
axes[2].imshow(pred_masked, cmap="jet", alpha=0.6, origin="lower", vmin=0, vmax=3)
axes[2].set_title("Model Prediction")
axes[2].axis("off")

fig.suptitle(f"{worst_patient_id} — Dice: {worst_5[0]['dice_overall']:.3f} — slice {best_slice}", fontsize=13)
plt.tight_layout()
save_path = os.path.join(PATHS["figures"], "worst_case_predictions.png")
plt.savefig(save_path, dpi=120, bbox_inches="tight")
plt.show()
print(f"Saved -> {save_path}")

In [ ]:
# Summary

print("NOTEBOOK 04 COMPLETE")
print("=" * 50)
print(f"Evaluated {len(results)} validation patients")
print()
print(f"Per-class Dice (mean):")
print(f"  NCR/NET: {np.mean(dice_ncr):.3f}  <- weakest class")
print(f"  ED:      {np.mean(dice_ed):.3f}")
print(f"  ET:      {np.mean(dice_et):.3f}")
print()
print(f"Key finding: tumor size has weak correlation with Dice ({correlation:.3f})")
print(f"             -> size is NOT the main driver of poor performance")
print()
print(f"Worst case ({worst_5[0]['patient_id']}, Dice={worst_5[0]['dice_overall']:.3f}):")
print(f"  Visual inspection shows the model confuses NCR/NET with ET")
print(f"  -> likely relies too heavily on FLAIR/T2 brightness patterns,")
print(f"     under-using T1ce contrast which best distinguishes NCR/NET")
print()
print(f"Hypothesis for next notebook (04b):")
print(f"  More training epochs (baseline stopped at 15, Dice was still rising)")
print(f"  may help the model learn class boundaries better, especially for")
print(f"  NCR/NET. Will retrain with more epochs and monitor per-class Dice.")
print()
print("Figures saved:")
for fname in ["worst_case_predictions.png"]:
    path = os.path.join(PATHS["figures"], fname)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{status}] {fname}")
print()
print("Next -> 04b_improvement.ipynb")
print("  - Retrain with more epochs (e.g. 40-50) and checkpoint/resume support")
print("  - Monitor per-class Dice, especially NCR/NET")
print("  - Compare against this baseline (Dice 0.657) to confirm improvement")